In [33]:
import pandas as pd
import ast
import numpy as np




pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [12]:
# ls

In [13]:
df = pd.read_csv("cross-encoder_Qwen-Qwen3-Reranker-4B.csv")

In [14]:
table = df["scores"].apply(ast.literal_eval).apply(pd.Series)
table

,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,high_school_chemistry,high_school_geography,jurisprudence,world_religions,formal_logic
0,-2.3125,-12.8125,-13.2500,-14.3125,-11.1250,-12.5000,-14.2500,-13.8125,-13.5625,-12.7500
1,-1.1875,-12.0000,-13.1875,-14.0625,-9.8125,-12.0000,-14.6875,-12.8750,-13.1875,-11.6250
2,5.1875,-11.3750,-12.0000,-13.4375,-8.6250,-11.0625,-14.2500,-11.8125,-10.9375,-5.4375
3,0.0000,-13.1875,-13.1250,-14.8750,-10.8750,-12.6250,-14.4375,-13.8125,-13.8125,-13.2500
4,-1.3750,-12.4375,-13.0000,-14.5625,-10.4375,-12.7500,-14.8125,-13.2500,-13.9375,-12.1875
...,...,...,...,...,...,...,...,...,...,...
196,-12.2500,-11.6875,-12.5000,-13.0000,-11.9375,-12.6875,-13.3125,-12.0000,-12.8125,-12.1250
197,-10.6250,-10.2500,-11.3750,-13.2500,-10.8125,-12.0000,-13.0000,-12.2500,-12.1250,-11.3750
198,-12.9375,-12.5000,-13.1250,-13.5000,-13.0625,-13.4375,-13.5625,-12.0000,-13.2500,-13.2500
199,-11.6875,-10.7500,-11.2500,-11.5625,-11.2500,-12.6250,-13.3750,-10.0000,-11.0000,-12.1250


In [15]:
middle = table.median()
tau = (table - middle).abs().median().mean()
print(tau)   

0.8875


In [16]:
is_general = (df["expected_service"] == "general").values

In [24]:
import random


# 1. chop specialists and put in a pool
def chop(pool):
    keep = is_general | df["expected_service"].isin(pool).values
    scores = table.loc[keep, pool]
    general = is_general[keep]
    return scores, general

In [34]:
scores, general = chop(["anatomy", "astronomy", "jurisprudence"])

print(scores.shape)     
print(general.sum())     
scores

(91, 3)
50


,anatomy,astronomy,jurisprudence
11,-9.18750,-11.06250,-10.93750
12,-0.12500,-12.87500,-14.25000
13,-4.18750,-11.37500,-12.56250
14,-0.62500,-12.06250,-13.31250
15,-5.75000,-12.62500,-13.37500
16,2.50000,-12.62500,-13.68750
17,-6.28125,-12.37500,-13.75000
18,-1.06250,-12.50000,-13.68750
19,-10.12500,-12.87500,-14.12500
20,-9.50000,-10.75000,-12.62500


In [35]:
trust = (scores.max(axis=1) - scores.mean(axis=1)) / tau
scores["trust"] = trust.round(2)
scores["is_general"] = general
scores

,anatomy,astronomy,jurisprudence,trust,is_general
11,-9.18750,-11.06250,-10.93750,1.36,False
12,-0.12500,-12.87500,-14.25000,10.09,False
13,-4.18750,-11.37500,-12.56250,5.85,False
14,-0.62500,-12.06250,-13.31250,9.06,False
15,-5.75000,-12.62500,-13.37500,5.45,False
16,2.50000,-12.62500,-13.68750,11.76,False
17,-6.28125,-12.37500,-13.75000,5.09,False
18,-1.06250,-12.50000,-13.68750,9.04,False
19,-10.12500,-12.87500,-14.12500,2.54,False
20,-9.50000,-10.75000,-12.62500,1.64,False


In [36]:
print(np.percentile(trust[general], 90))

1.1784037558685443


In [37]:
random.seed(0)
names = list(table.columns)

rows = []
for k in range(3, 11):
    cutoffs = []
    for try_number in range(60):
        pool = random.sample(names, k)
        scores, general = chop(pool)           # a fresh chop each time

        trust = (scores.max(axis=1) - scores.mean(axis=1)) / tau
        cutoffs.append(np.percentile(trust[general], 90))

    rows.append({"K": k,
                 "cutoff (average)": np.mean(cutoffs),
                 "lowest pool": np.min(cutoffs),
                 "highest pool": np.max(cutoffs)})

targets = pd.DataFrame(rows)
targets.round(2)

,K,cutoff (average),lowest pool,highest pool
0,3,1.28,0.85,1.90
1,4,1.49,1.01,1.96
2,5,1.61,1.27,2.09
3,6,1.71,1.37,2.06
4,7,1.80,1.52,2.06
5,8,1.86,1.64,2.12
6,9,1.89,1.75,2.01
7,10,1.91,1.91,1.91
